[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/rendering/examples/fluids/dec_stokes/dec_stokes.ipynb)

# Stokes Flow and Discrete Exterior Calculus

Stokes flow balances pressure against viscous resistance when inertia is negligible. This example finds the pressure needed to carry fluid through a channel around an obstacle. Every triangle passes on exactly as much fluid as it receives; the walls bring the fluid to rest.

Integrated fluxes are scalar fields over mesh edges. Incidence maps sum these signed readings around cells, while diagonal Hodge maps supply lengths, areas and complementary orientation. Their composition gives viscous resistance, and pressure enforces conservation on every triangle.

In differential-form notation, the topological identity is $d_1d_0=0$, and the circumcentric Hodge entries are $|\star\sigma^k|/|\sigma^k|$. [Jagad et al.](https://arxiv.org/abs/2010.15520) give a pressure–velocity DEC formulation of incompressible flow.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga@rendering"

In [ ]:
%matplotlib inline
import numpy as np

from numga.algebras import VGA2D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.fluids.dec_stokes import render, scenarios
from examples.fluids.dec_stokes.numerical import solve_stokes
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, mv = mesh_module.Mesh, mesh_module.mv
Scalar = ga.gatype.scalar()
Point = ga.gatype.antivector()
Amount = ga.gatype.antiscalar()

np.set_printoptions(precision=4, suppress=True)

## The channel

The obstacle leaves unequal gaps above and below it. A prescribed throughput will divide between them, with the pressure drop determined by viscous resistance. Lengths are measured in half-channel widths, speeds in the mean inlet speed, and viscosity is one.

In [ ]:
n_divisions = 18
viscosity, mean_speed = 1.0, 1.0
height = 2.0
n_frames, duration_ms = 48, 70
mesh, vertices, inlet, outlet, walls = scenarios.channel(n_divisions)

save_figure(render.draw_mesh(mesh, vertices), "dec_stokes_mesh");

## Cochains and incidence

A cochain holds one signed integral per cell: vertex readings, edge integrals or face integrals. These all fit in scalar fields, with the mesh's vertices, edges and faces as their field axes. Form degree belongs to the cell family; it is not an ambient blade grade. The map types track the scalar values and field sizes, while the mesh supplies their meaning. Incidence contains only signs: `d1(d0)` vanishes because each vertex occurs twice with opposite signs in a triangle's boundary.

In [ ]:
d0 = mesh.d0 * Scalar                                      # Scalar[E] <- Scalar[V]
d1 = mesh.d1 * Scalar                                      # Scalar[F] <- Scalar[E]

print("Vertex to edge:", d0.gatype, d0.shape)
print("Edge to face:  ", d1.gatype, d1.shape)

## Primal and dual cells

Each triangle has a dual vertex at its circumcentre, where the distances to its corners agree. Neighbouring circumcentres define the dual edge crossing their shared primal edge. Around a primal vertex these edges enclose a dual polygon. The primal mesh is blue and its dual orange; the highlighted triangle and polygon show the two kinds of area.

In [ ]:
edges = mesh.d0 * vertices                                 # Point[E]
corners = vertices.batch()[mesh.faces.T].field()            # [corners] Point[F]
a, b, c = corners
u, v = b - a, c - a
area_blades = (u & v) / 2                                  # Antibivector[F]
areas = area_blades.norm()                                 # Scalar[F]
sense = (area_blades / areas).sites.mean()                  # Antibivector
face_centres = corners.sum(axis=0) / 3                      # Point[F]

# The triangle's area turns the squared-length-weighted edge difference
# into the offset equidistant from all three corners.
weighted = v.norm_squared() * u - u.norm_squared() * v
circumcentres = a + weighted.commutator(area_blades.dual()) / (4 * areas.squared())

one_face = mv.scalar(np.ones((len(mesh.faces), 1))).field()
tail, head = mesh.edge_ends
midpoints = (tail * vertices + head * vertices) / 2          # Point[E]
boundary = ~mesh.d1 * one_face                             # Scalar[E]
# Interior dual edges join adjacent circumcentres. At the boundary they
# finish at the primal edge's midpoint, closing the truncated dual cells.
# The dual edge crosses to the primal edge's right, matching positive flux.
dual_edges = boundary * midpoints - ~mesh.d1 * circumcentres

save_figure(render.draw_complex(mesh, vertices, circumcentres, dual_edges), "dec_stokes_complex");

## Diagonal Hodge maps

The Hodge maps relate readings on primal cells to readings on their complementary dual cells. Their weights are dual areas at vertices, dual-to-primal length ratios at edges, and reciprocal areas at faces. Each also changes the value's orientation, from `Scalar` to `Amount`, so primal and dual readings pair through `&`. Algebraic duality supplies this change of carrier; the mesh supplies which cells correspond, and geometry supplies their measures.

In [ ]:
# Signed dual/primal edge-length ratios, and the dual areas made of
# the quarter-kites incident to each vertex.
h1 = (dual_edges & edges) / (sense * edges.norm_squared())
dual_areas = (~tail + ~head) * (h1 * edges.norm_squared() / 4)

star0 = spdiag(dual_areas * Scalar.dual())                   # Amount[V] <- Scalar[V]
star1 = spdiag(h1 * Scalar.dual())                           # Amount[E] <- Scalar[E]
star2 = spdiag(Scalar.dual() / areas)                        # Amount[F] <- Scalar[F]
inverse_star0 = spdiag(Amount.dual_inverse() / dual_areas)    # Scalar[V] <- Amount[V]

print("Primal to dual readings:", star1.gatype)
print("Dual incidence:        ", d0.adjugate().gatype, d0.adjugate().shape)

## Vorticity and viscous resistance

The dual of an edge flux is circulation along the crossing dual edge. Summing these circulations around a dual polygon and dividing by its area gives vorticity. The incidence's adjugate performs that sum on the complementary fields: `d0(potential) & circulation` and `potential & d0.adjugate()(circulation)` give the same total pairing. Vorticity differences drive the viscous resistance to flow.

In [ ]:
curl = inverse_star0(d0.adjugate()(star1))                   # Scalar[V] <- Scalar[E]
viscous = viscosity * star1(d0(curl))                        # Amount[E] <- Scalar[E]

## Boundary flow and pressure

Pressure enforces zero net flux through every triangle. The inlet and outlet carry the same parabolic profile. On the solid walls, zero normal flux and zero tangential boundary circulation impose no slip. Fluxes and pressures are solved together; the arbitrary pressure offset is fixed by its area-weighted mean.

In [ ]:
# The antiderivative of the prescribed inlet profile gives each boundary edge's flux.
ordinate = mv.y & vertices
inlet_potential = 1.5 * mean_speed * (ordinate - 4 * ordinate.squared() * ordinate / (3 * height**2))
ends = SparseExtensor.selection(mv.context, np.concatenate((inlet, outlet)), len(mesh.edges))
prescribed_flux = ~ends * (ends * d0(inlet_potential))
fixed_edges = mesh.boundary_edges
boundary_circulation = mv.scalar(np.zeros((mesh.n_vertices, 1))).field()

# Boundary segments close the dual polygons; their prescribed circulation
# contributes to vorticity and hence to viscous resistance.
boundary_vorticity = boundary_circulation / dual_areas
boundary_resistance = viscosity * star1(d0(boundary_vorticity))

free_edges = np.setdiff1d(np.arange(len(mesh.edges)), fixed_edges)
embedding = ~SparseExtensor.selection(mv.context, free_edges, len(mesh.edges)) * Scalar
stiffness = embedding.adjugate()(viscous(embedding))          # Amount[free] <- Scalar[free]
divergence = d1(embedding)                                  # Scalar[F] <- Scalar[free]
force = -embedding.adjugate()(viscous(prescribed_flux) + boundary_resistance)
balance = -d1(prescribed_flux)                              # Scalar[F]
unknown, pressure = solve_stokes(stiffness, divergence, force, balance)
flux = prescribed_flux + embedding(unknown)                 # Scalar[E]
pressure = pressure - (pressure * areas).sites.sum() / areas.sites.sum()

# The mesh's oriented area turns the signed vorticity reading into its plane.
vorticity = sense.dual_inverse() * (curl(flux) + boundary_vorticity)
# The first moment of the outward flux gives the face's mean velocity.
outward = (flux.batch()[mesh.face_edges.T] * mesh.face_edge_orientation.T).field()
velocity = -((corners - face_centres) * outward).sum(axis=0) / (2 * areas)

# A streamfunction draws the conserved flux's paths on the native triangles.
streamfunction = d0.lstsq(flux)
save_figure(render.draw_fields(mesh, vertices, pressure, vorticity, streamfunction), "dec_stokes_fields");

The upper panel shows pressure, falling from warm to cool colours across the obstacle. The lower panel shows the two senses of vorticity, concentrated where the flow shears past the walls and obstacle. The same streamlines cross both panels; their spacing records how the throughput divides.

In [ ]:
inlet_read = SparseExtensor.selection(mv.context, inlet, len(mesh.edges))
throughput = -(inlet_read * (boundary * flux)).sites.sum()
inlet_faces = np.flatnonzero(np.isin(mesh.face_edges, inlet).any(axis=1))
outlet_faces = np.flatnonzero(np.isin(mesh.face_edges, outlet).any(axis=1))
pressure_drop = pressure.batch()[inlet_faces].mean() - pressure.batch()[outlet_faces].mean()
print("Throughput per unit depth:", throughput.to_array())
print("Pressure drop near the ends:", pressure_drop.to_array())
print("Largest cell flux imbalance:", abs(d1(flux).to_array()).max())

## Flow paths

Tracers follow the steady flow, with their travel times set by the computed velocity. They follow the streamfunction contours on the triangles, including the constant contour along the obstacle boundary.

In [ ]:
show_animation(render.animate_flow(mesh, vertices, pressure, streamfunction, velocity, n_frames),
               "dec_stokes_flow", duration_ms, n_frames=n_frames)

## Reversing edge directions

Reversing an edge changes its signed integral and the incidences that refer to it. The corresponding dual edge reverses too, leaving its Hodge weight unchanged. Cell balances and vorticity therefore agree after either choice of edge direction. Intrinsic and extrinsic orientation specify the kind of value; a mesh edge's direction specifies the sign of a particular reading.

In [ ]:
# Reverse every third edge, together with the primal and dual readings on it.
edge_signs = mv.scalar(np.where(np.arange(len(mesh.edges)) % 3 == 0, -1, 1)[:, None]).field()
reorient = spdiag(edge_signs * Scalar)
reversed_d0 = reorient(d0)
reversed_d1 = d1(reorient)
reversed_star1 = reorient.adjugate()(star1(reorient))
reversed_flux = reorient(flux)
reversed_curl = inverse_star0(reversed_d0.adjugate()(reversed_star1))

print("Change in cell balances:", abs((reversed_d1(reversed_flux) - d1(flux)).to_array()).max())
print("Change in vorticity:", abs((reversed_curl(reversed_flux) - curl(flux)).to_array()).max())

In [ ]:
# checks
potential = (mv.x & vertices).squared() + 0.3 * (mv.y & vertices)
np.testing.assert_allclose(d1(d0(potential)).to_array(), 0, atol=1e-9)
np.testing.assert_allclose(d1(flux).to_array(), 0, atol=1e-9)
np.testing.assert_allclose(reversed_d1(reversed_flux).to_array(), 0, atol=1e-9)
np.testing.assert_allclose((reversed_curl(reversed_flux) - curl(flux)).to_array(), 0, atol=1e-9)